In [0]:
#Leyendo la tabla de caso practico
cpv= spark.read.format("delta").table("workspace.electiva.caso_practico_ventas")
# Convertir el DataFrame de Spark a Pandas
cpv = cpv.toPandas()

In [0]:
cpv

### Bloque 1: Exploración y Diagnóstico (Data Profiling)

Carga e Inspección: Importe el archivo CSV y visualice las primeras 10 y últimas 5 filas. Verifique el total de registros y
columnas.

In [0]:
cpv.head(10)
cpv.tail(5)
cpv.shape

Identificación de Tipos: Liste los tipos de datos asignados automáticamente por la herramienta. ¿Coinciden con los
esperados?

In [0]:
cpv.dtypes

In [0]:
cpv.info()

La herramienta asignó automáticamente los tipos de datos de la siguiente manera: 6 variables fueron identificadas como object (texto) y 3 como float64 (numéricas). En general, los tipos coinciden con los esperados para las variables, aunque la columna fecha fue identificada como object y debería convertirse a un formato de fecha (datetime) para facilitar su análisis.

**Detección de Nulos: Cuantifique el número total y porcentaje de valores nulos o faltantes para cada columna.**

In [0]:
import pandas as pd

nulos = cpv.isnull().sum()
porcentaje_nulos = (cpv.isnull().sum() / len(cpv)) * 100

resultado_nulos = pd.DataFrame({
    'Total nulos': nulos,
    'Porcentaje nulos': porcentaje_nulos
})

resultado_nulos

In [0]:
cpv[['cantidad', 'precio_unitario']].describe()

In [0]:
estadisticas = pd.DataFrame({
    'Media': cpv[['cantidad', 'precio_unitario']].mean(),
    'Mediana': cpv[['cantidad', 'precio_unitario']].median(),
    'Desviación estándar': cpv[['cantidad', 'precio_unitario']].std(),
    'Mínimo': cpv[['cantidad', 'precio_unitario']].min(),
    'Máximo': cpv[['cantidad', 'precio_unitario']].max()
})

estadisticas

### Bloque 2: Limpieza y Depuración (Data Cleaning)

In [0]:
duplicados_completos = cpv[cpv.duplicated(keep=False)]
print("Filas completamente duplicadas:")
print(duplicados_completos)

duplicados_id = cpv[cpv.duplicated(subset='id_transaccion', keep=False)]
print("Registros con id_transaccion repetido:")
print(duplicados_id)

cpv = cpv.drop_duplicates()
cpv = cpv.drop_duplicates(subset='id_transaccion', keep='first')


print("Cantidad de filas después de eliminar duplicados:", len(cpv))

Normalización de Texto: En metodo_pago, reemplace las variantes de "Tarjeta Credito" para unificarlas como "Tarjeta de
Crédito".

In [0]:
cpv['metodo_pago'] = cpv['metodo_pago'].replace(
    ['Tarjeta Credito', 'tarjeta credito', 'Tarjeta crédito', 'tarjeta crédito'],
    'Tarjeta de Crédito'
)

In [0]:
print(cpv['metodo_pago'].unique())

**Gestión de Valores Nulos**

Reemplace los valores nulos en cantidad por la mediana de la columna.

In [0]:
cpv['cantidad'] = cpv['cantidad'].fillna(cpv['cantidad'].median())
print(cpv['cantidad'].isnull().sum())

Impute los faltantes en satisfaccion_cliente usando la moda (valor más frecuente)

In [0]:
cpv['satisfaccion_cliente'] = cpv['satisfaccion_cliente'].fillna(
    cpv['satisfaccion_cliente'].mode()[0]
)

In [0]:
print(cpv['satisfaccion_cliente'].isnull().sum())

Tratamiento de Atípicos (Outliers): Localice la transacción con precio_unitario = 9999.00. Decida si eliminar la fila o
imputarle el precio medio según su categoría.

In [0]:
atipico = cpv[cpv['precio_unitario'] == 9999.00]

print("Transacción atípica:")
print(atipico)

categoria = atipico['categoria_producto'].iloc[0]
precio_medio = cpv[
    (cpv['categoria_producto'] == categoria) &
    (cpv['precio_unitario'] != 9999.00)
]['precio_unitario'].mean()

print("Precio medio de la categoría:", precio_medio)

cpv.loc[cpv['precio_unitario'] == 9999.00, 'precio_unitario'] = precio_medio

print(cpv[cpv['categoria_producto'] == categoria][
    ['categoria_producto', 'precio_unitario']
])

Se identificó la transacción con un precio unitario de $9.999,00, considerado un valor atípico. En lugar de eliminar la fila completa, se decidió imputar el precio utilizando el promedio de los precios de su categoría, conservando así la información de la transacción.

### Bloque 3: Transformación e Ingeniería de Atributos

Columna Derivada: Calcule la variable monto_total mediante el producto vectorial: monto_total = cantidad *
precio_unitario.

In [0]:
cpv['monto_total'] = cpv['cantidad'] * cpv['precio_unitario']

print(cpv[['cantidad', 'precio_unitario', 'monto_total']].head())

Conversión Temporal: Transforme la columna fecha a un objeto tipo fecha/tiempo verdadero y extraiga una nueva columna
con el dia_semana o mes.

In [0]:
cpv['fecha'] = pd.to_datetime(cpv['fecha'])
cpv['dia_semana'] = cpv['fecha'].dt.day_name()
print(cpv[['fecha', 'dia_semana']].head())

In [0]:
dias_es = {
    'Monday': 'Lunes',
    'Tuesday': 'Martes',
    'Wednesday': 'Miércoles',
    'Thursday': 'Jueves',
    'Friday': 'Viernes',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}
cpv['dia_semana'] = cpv['fecha'].dt.day_name().map(dias_es)

Discretización (Binning): Agrupe las transacciones según su monto_total en 3 rangos: 
- Bajo: Monto < $100 
- Medio: Monto entre $100 y $500 
- Alto: Monto > $500

In [0]:
import numpy as np

In [0]:
condiciones = [
    cpv['monto_total'] < 100,
    (cpv['monto_total'] >= 100) & (cpv['monto_total'] <= 500),
    cpv['monto_total'] > 500
]

categorias = ['Bajo', 'Medio', 'Alto']

cpv['rango_monto'] = np.select(condiciones, categorias, default='')
print(cpv[['monto_total', 'rango_monto']].head())

### Bloque 4: Agregación y Resumen de Datos

Agrupamiento Simple (Group By): Obtenga el total de ingresos (suma de monto_total) y el promedio de unidades vendidas
agrupado por categoria_producto.

In [0]:
resumen_categoria = cpv.groupby('categoria_producto').agg(
    ingresos_totales=('monto_total', 'sum'),
    promedio_unidades=('cantidad', 'mean')
).reset_index()

print(resumen_categoria)

Tabla Dinámica (Pivot Table): Genere una tabla cruzada que muestre la suma total de monto_total colocando la region en
filas y categoria_producto en columnas.

In [0]:
tabla_dinamica = pd.pivot_table(
    cpv,
    values='monto_total',
    index='region',
    columns='categoria_producto',
    aggfunc='sum',
    fill_value=0
)


print(tabla_dinamica)

Análisis de Satisfacción: Calcule el nivel medio de satisfacción de los clientes agrupado por metodo_pago

In [0]:
satisfaccion_pago = cpv.groupby('metodo_pago')['satisfaccion_cliente'].mean()
print(satisfaccion_pago)

### Bloque 5: Selección, Filtrado y Muestreo

Filtrado Multicriterio: Filtre la tabla para conservar únicamente las ventas de la región Norte con una satisfacción igual o
superior a 4.

In [0]:
ventas_norte = cpv[
    (cpv['region'] == 'Norte') &
    (cpv['satisfaccion_cliente'] >= 4)
]

print(ventas_norte)

Muestreo Estratificado: Extraiga una muestra aleatoria equivalente al 10% de los registros conservando la proporción por
region.

In [0]:
muestra_10 = cpv.groupby('region', group_keys=False).sample(
    frac=0.10,
    random_state=42
)

print(muestra_10)
print(muestra_10['region'].value_counts())